# Eval ????? (??????)
??????? ???????? ??????????? answer_eval: baseline (101), gold (50), Ollama sample20/gold50.


In [0]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')
sns.set_palette('colorblind')

ROOT = Path(__file__).resolve().parent.parent  # eval/
EVAL_DIR = ROOT
FILES = {
    'deepseek': EVAL_DIR / 'answer_eval_results.jsonl',
    'deepseek_gold': EVAL_DIR / 'answer_eval_results_gold.jsonl',
    'ollama': EVAL_DIR / 'answer_eval_results_ollama_sample20.jsonl',
    'ollama_gold': EVAL_DIR / 'answer_eval_results_gold_ollama.jsonl',
}
for k, p in FILES.items():
    print(k, '->', p, 'exists' if p.exists() else 'MISSING')


In [0]:
def load_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    rows = []
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))
    return pd.DataFrame(rows)

df_main = load_jsonl(FILES['deepseek'])
df_gold = load_jsonl(FILES['deepseek_gold'])
df_ollama = load_jsonl(FILES['ollama'])
df_ollama_gold = load_jsonl(FILES['ollama_gold'])

print('DeepSeek shape:', df_main.shape)
print('DeepSeek gold shape:', df_gold.shape)
print('Ollama shape:', df_ollama.shape)
print('Ollama gold shape:', df_ollama_gold.shape)


In [0]:
def show_head(df: pd.DataFrame, label: str):
    if df.empty:
        print(label + ': ??? ??????')
        return
    display(df.head()[['question','answer','answer_ref']])

show_head(df_main, 'DeepSeek')
show_head(df_gold, 'DeepSeek gold (50)')
show_head(df_ollama, 'Ollama')
show_head(df_ollama_gold, 'Ollama gold (50)')


In [0]:
def summarize(df: pd.DataFrame, label: str):
    if df.empty:
        print(label + ': ??? ??????')
        return
    cols = [c for c in ['ref_sim','refs_sim'] if c in df.columns]
    if not cols:
        print(label + ': ??? ???????? ref_sim/refs_sim')
        return
    df_num = df[cols].apply(pd.to_numeric, errors='coerce')
    summary = df_num.agg(['mean','median','std']).T
    summary = summary.reset_index().rename(columns={'index':'metric'})
    summary = summary[['metric','mean','median','std']]
    display(summary.style.format({'mean':'{:.4f}','median':'{:.4f}','std':'{:.4f}'}).set_caption(label))

summarize(df_main, 'DeepSeek (101)')
summarize(df_gold, 'DeepSeek gold (50)')
summarize(df_ollama, 'Ollama qwen2.5 (20)')
summarize(df_ollama_gold, 'Ollama qwen2.5 gold (50)')


In [0]:
def plot_hists(df: pd.DataFrame, title: str):
    if df.empty:
        print(title + ': ??? ??????')
        return
    cols = [c for c in ['ref_sim','refs_sim'] if c in df.columns]
    if not cols:
        print(title + ': ??? ???????? ref_sim/refs_sim')
        return
    df_num = df[cols].apply(pd.to_numeric, errors='coerce')
    df_num.plot(kind='hist', bins=20, alpha=0.7, title=title)
    plt.xlabel('similarity')
    plt.show()

plot_hists(df_main, 'DeepSeek: ????????????? sim')
plot_hists(df_gold, 'DeepSeek gold: ????????????? sim')
plot_hists(df_ollama, 'Ollama: ????????????? sim')
plot_hists(df_ollama_gold, 'Ollama gold: ????????????? sim')
